# Download Data

This notebook downloads and validates the SymbTr v3.0 Turkish Maqam Music Dataset from Zenodo.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "webbook" / "data" / "raw"

DATA_RAW.mkdir(parents=True, exist_ok=True)

print(DATA_RAW)

In [20]:
import requests

record_id = 15470412

url = f"https://zenodo.org/api/records/{record_id}"

r = requests.get(url)
r.raise_for_status()

data = r.json()

for f in data["files"]:
    print(f["key"])

mid_v3.zip
Turkish maqam music pieces in time series format_3000 rows x 365 columns.csv
txt_v3.zip
mu2_v3.zip
xml_v3.zip
pdf_v3.zip


In [21]:
import requests
from urllib.parse import quote

filename = "Turkish maqam music pieces in time series format_3000 rows x 365 columns.csv"

url = "https://zenodo.org/records/15470412/files/" + quote(filename) + "?download=1"

target_file = DATA_RAW / filename

response = requests.get(url, stream=True)
response.raise_for_status()

with open(target_file, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print("Saved:", target_file)

Saved: C:\Users\tualtan\Projects\SymbTr-Analysis-Book\webbook\data\raw\Turkish maqam music pieces in time series format_3000 rows x 365 columns.csv


In [22]:
print(target_file.exists())
print(round(target_file.stat().st_size / (1024 * 1024), 2), "MB")

True
4.27 MB


In [23]:
import pandas as pd

df = pd.read_csv(target_file)

print(df.shape)
df.head()

(3000, 366)


,ID,p1,p2,p3,p4,p5,p6,p7,p8,p9,...,p356,p357,p358,p359,p360,p361,p362,p363,p364,p365
0,acem--ilahi--duyek--aldanma_dunya--zekai_dede,318.0,318.0,318.0,340.0,340.0,336.0,340.0,340.0,336.0,...,318.0,313.0,313.0,305.0,296.0,318.0,305.0,305.0,305,305
1,acem--ilahi--nimevsat--calabim_bir--haci_bayra...,305.0,305.0,305.0,340.0,340.0,340.0,340.0,340.0,340.0,...,313.0,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305,305
2,acem--kupe--duyek--zulfunu--ahmet_avni_konuk,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305.0,...,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305,305
3,acem--selam--devrikebir--asik-i_ger--huseyin_f...,318.0,340.0,349.0,340.0,340.0,349.0,327.0,313.0,318.0,...,318.0,313.0,305.0,305.0,318.0,318.0,287.0,305.0,287,287
4,acem--seyir--sofyan----sefik_gurmeric,305.0,305.0,305.0,305.0,305.0,305.0,340.0,340.0,340.0,...,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305.0,305,305


In [1]:
from pathlib import Path
import re
import requests
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
WEBBOOK_ROOT = PROJECT_ROOT / "webbook"

INCLUDE_DIR = WEBBOOK_ROOT / "_includes"
INCLUDE_DIR.mkdir(parents=True, exist_ok=True)

ZENODO_URL = "https://zenodo.org/records/15470412"

response = requests.get(ZENODO_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
text = soup.get_text(" ", strip=True)

patterns = {
    "pieces": r"consists of\s+([\d,\.]+)\s+pieces",
    "makams": r"from\s+([\d,\.]+)\s+makams",
    "usuls": r"([\d,\.]+)\s+usuls",
    "forms": r"([\d,\.]+)\s+forms",
    "notes": r"about\s+([\d,\.]+)\s+musical notes",
    "hours": r"([\d,\.]+)\s+hours nominal playback time",
}

stats = {}

for key, pattern in patterns.items():
    match = re.search(pattern, text, flags=re.IGNORECASE)
    stats[key] = match.group(1) if match else "N/A"

formats = ["MusicXML", "MIDI", "PDF", "text", "mu2"]

markdown = f"""
## SymbTr Collection Overview

The latest version of the SymbTr collection consists of:

- Approximately **{stats["pieces"]} musical pieces**
- **{stats["makams"]} makams** (modal structures)
- **{stats["usuls"]} usuls** (rhythmic cycles)
- **{stats["forms"]} musical forms**
- About **{stats["notes"]} musical notes**
- Roughly **{stats["hours"]} hours** of nominal playback time

The collection includes scores in multiple formats such as {", ".join(formats)}, making it suitable for computational analysis and music software applications.

_Source: SymbTr v3.0 Zenodo record._
""".strip()

output_file = INCLUDE_DIR / "symbtr_stats.md"
output_file.write_text(markdown, encoding="utf-8")

print(markdown)
print("\nSaved to:", output_file)

## SymbTr Collection Overview

The latest version of the SymbTr collection consists of:

- Approximately **3000 musical pieces**
- **164 makams** (modal structures)
- **135 usuls** (rhythmic cycles)
- **61 musical forms**
- About **1.200.000 musical notes**
- Roughly **145 hours** of nominal playback time

The collection includes scores in multiple formats such as MusicXML, MIDI, PDF, text, mu2, making it suitable for computational analysis and music software applications.

_Source: SymbTr v3.0 Zenodo record._

Saved to: C:\Users\tualtan\Projects\SymbTr-Analysis-Book\webbook\_includes\symbtr_stats.md
